In [1]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Activation
from tensorflow.keras.optimizers import RMSprop


In [2]:
filepath = tf.keras.utils.get_file('shakespeare.txt','https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(filepath, 'rb').read().decode(encoding='utf-8').lower()


In [5]:
text = text[0:500000]
characters = sorted(set(text))
print("Text length:", len(text))
print("Vocabulary size:", len(characters))
print("First 100 characters:", repr(text[:100]))
char_to_index = dict((c, i) for i, c in enumerate(characters))
index_to_char = dict((i, c) for i, c in enumerate(characters))
SEQ_LENGTH = 40
STEP_SIZE = 3

Text length: 500000
Vocabulary size: 37
First 100 characters: ' hast spoken words?\n\nsicinius:\no blessed heavens!\n\nvolumnia:\nmore noble blows than ever thou wise wo'


In [6]:
sentences = []
next_characters = []

In [7]:
for i in range (0, len(text) - SEQ_LENGTH, STEP_SIZE):
    sentences.append(text[i: i + SEQ_LENGTH])
    next_characters.append(text[i + SEQ_LENGTH])

In [8]:
x = np.zeros((len(sentences), SEQ_LENGTH, len(characters)), dtype = np.bool)
y = np.zeros((len(sentences), len(characters)), dtype = np.bool)


for i,sentence in enumerate(sentences):
    for t, character in enumerate(sentence):
        x[i, t, char_to_index[character]] = 1
    y[i, char_to_index[next_characters[i]]] = 1

In [ ]:
model = Sequential()
model.add(LSTM(128, input_shape = (SEQ_LENGTH, len(characters))))
model.add(Dense(len(characters)))
model.add(Activation('softmax'))
model.compile(loss = 'categorical_crossentropy', optimizer = RMSprop(learning_rate = 0.01))
model.fit(x, y, batch_size = 256, epochs = 6)

model.save('textgenerator.keras')

#changes that can be applied:
'''
Architecture: one LSTM(128) became two stacked LSTM(256) layers, with return_sequences=True on the first so it feeds the second.
Dropout: added Dropout(0.2) after each LSTM to reduce overfitting.
Softmax: the separate Dense + Activation('softmax') is now a single Dense(..., activation='softmax'). Same behavior, less code.
Optimizer: RMSprop(learning_rate=0.01) became Adam(learning_rate=0.001, clipnorm=1.0). The lower learning rate makes training more stable, 
and gradient clipping prevents exploding gradients.
Callbacks: added ModelCheckpoint (saves the best model), EarlyStopping (stops when validation loss stops improving), 
and ReduceLROnPlateau (halves the learning rate when progress stalls).
Training: epochs went from 6 to 50 (early stopping will usually end it sooner), and validation_split=0.1 was added so the callbacks have a val_loss to monitor.
'''
'''
model = Sequential([
    LSTM(256, return_sequences=True, input_shape=(SEQ_LENGTH, len(characters))),
    Dropout(0.2),
    LSTM(256),
    Dropout(0.2),
    Dense(len(characters), activation='softmax'),
])

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
)

callbacks = [
    ModelCheckpoint('textgenerator.keras', monitor='val_loss', save_best_only=True),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

model.fit(x, y, batch_size=256, epochs=50, validation_split=0.1, callbacks=callbacks)

model.save('textgenerator.keras')
'''

Epoch 1/6
651/651 ━━━━━━━━━━━━━━━━━━━━ 59s 89ms/step - loss: 2.2339
Epoch 2/6
651/651 ━━━━━━━━━━━━━━━━━━━━ 59s 91ms/step - loss: 1.7529
Epoch 3/6
651/651 ━━━━━━━━━━━━━━━━━━━━ 60s 92ms/step - loss: 1.6013
Epoch 4/6
651/651 ━━━━━━━━━━━━━━━━━━━━ 73s 113ms/step - loss: 1.5217
Epoch 5/6
651/651 ━━━━━━━━━━━━━━━━━━━━ 84s 128ms/step - loss: 1.4709
Epoch 6/6
651/651 ━━━━━━━━━━━━━━━━━━━━ 77s 119ms/step - loss: 1.4339


In [46]:
pip install tf_keras


Note: you may need to restart the kernel to use updated packages.


In [11]:
model = tf.keras.models.load_model('textgenerator.keras')
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(model, length, temperature):
    start_index = np.random.randint(0, len(text) - SEQ_LENGTH - 1)
    generated = ''
    sentence = text[start_index: start_index + SEQ_LENGTH]
    generated += sentence
    for i in range(length):
        x = np.zeros((1, SEQ_LENGTH, len(characters)))
        for t, character in enumerate(sentence):
            x [0, t, char_to_index[character]] = 1.

        predictions = model.predict(x, verbose=0)[0]
        next_index = sample(predictions, temperature)
        next_character = index_to_char[next_index]

        generated += next_character
        sentence = sentence[1:] + next_character
    return generated

print(generate_text(model, length=300, temperature=0.2))
       



ack, poor richard! where rode he the which in hereford!

henry bolingbroke:
and stay he with the sease to the senore
the sease to the seass of heavy to the prince.

king hishamow:
then i will be the seased to the seased.

buckingham:
and i will be hear me to the seased,
thou hast not speak a proved the prince and blood
the see the sease t
